# 🎭 Aarya on Google Colab — ComfyUI identity pack
Runs the Aarya workflows from `dhanushpatilv388-hue/blank-app` (aarya/ folder) on Colab.

**Tier logic (automatic):** cell 1 reads your GPU VRAM.
- **≥ 20 GB** (L4 / A100 — Colab Pro/Pro+ draws): downloads the Wan-Animate-2 stack → motion video + stills.
- **< 20 GB** (free-tier T4): downloads the face-lock stills stack only. Motion video on a T4 is not
  practical (14B int8 ≈ 15.5 GB weights + T5/CLIP/VAE); use a Pro L4/A100 session or the Runpod runbook.

**Persistence:** all models live in `Drive/MyDrive/aarya/models` — downloaded ONCE, reused every session.
**Session death:** outputs are copied to Drive by the last cell; keep this tab focused to avoid idle-kick.


In [ ]:
# 1) GPU check + tier
import subprocess, os
out = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'],
                     capture_output=True, text=True).stdout.strip()
print(out)
vram = int(out.split(',')[-1].strip())
TIER = 'A' if vram >= 20000 else 'B'
print(f"VRAM {vram} MiB -> TIER {TIER} " + ("(Wan-Animate-2 video + stills)" if TIER=='A' else "(stills only; video needs >=20GB: Pro L4/A100 or Runpod)"))


In [ ]:
# 2) Mount Google Drive (models persist across sessions)
from google.colab import drive
drive.mount('/content/drive')
ROOT='/content/drive/MyDrive/aarya'
for d in ['models/checkpoints','models/loras','models/text_encoders','models/clip_vision','models/vae',
          'models/inswapper','models/facerestore_models','workflows','input','output']:
    os.makedirs(f'{ROOT}/{d}', exist_ok=True)
print('Drive ready at', ROOT)


In [ ]:
# 3) ComfyUI + deps
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone --depth 1 https://github.com/Comfy-Org/ComfyUI.git /content/ComfyUI
os.chdir('/content/ComfyUI')
!pip install -q -r requirements.txt
# model dirs -> Drive (one-time downloads survive sessions)
import os
for sub in ['checkpoints','loras','text_encoders','clip_vision','vae','inswapper','facerestore_models']:
    src=f'/content/ComfyUI/models/{sub}'; dst=f'{ROOT}/models/{sub}'
    if os.path.islink(src): os.remove(src)
    if os.path.isdir(src) and not os.path.islink(src):
        import shutil; shutil.rmtree(src)
    os.symlink(dst, src)
print('ComfyUI ready, model dirs symlinked to Drive')


In [ ]:
# 4) Models (tier-aware, resumable, skip-if-present)
import os
A = [  # Wan-Animate-2 stack (~24 GB) - TIER A only
 ('text_encoders','umt5_xxl_fp8_e4m3fn_scaled.safetensors','https://huggingface.co/Comfy-Org/Wan-Animate-2/resolve/main/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors'),
 ('clip_vision','clip_vision_h.safetensors','https://huggingface.co/Comfy-Org/Wan-Animate-2/resolve/main/clip_vision/clip_vision_h.safetensors'),
 ('vae','Wan2_1_VAE_bf16.safetensors','https://huggingface.co/Comfy-Org/Wan-Animate-2/resolve/main/vae/Wan2_1_VAE_bf16.safetensors'),
 ('loras','lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors','https://huggingface.co/Comfy-Org/Wan-Animate-2/resolve/main/loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors'),
 ('diffusion_models','wan_animate_2_int8_convrot.safetensors','https://huggingface.co/Comfy-Org/Wan-Animate-2/resolve/main/diffusion_models/wan_animate_2_int8_convrot.safetensors'),
]
B = [  # face-lock stills stack (~3.5 GB) - both tiers
 ('checkpoints','v1-5-pruned-emaonly-fp16.safetensors',['https://huggingface.co/Comfy-Org/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly-fp16.safetensors','https://huggingface.co/sd-legacy/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly-fp16.safetensors']),
 ('loras','lcm-lora-sdv1-5.safetensors',['https://huggingface.co/latents/lcm-lora/resolve/main/lcm-lora-sdv1-5.safetensors']),
 ('inswapper','inswapper_128.onnx',['https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx']),
 ('facerestore_models','codeformer-v0.1.0.pth',['https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer-v0.1.0.pth']),
]
A = [(s,f,[u]) for s,f,u in A]
B = [(s,f,(u if isinstance(u,list) else [u])) for s,f,u in B]
jobs = B + (A if TIER=='A' else [])
for sub, fn, urls in jobs:
    dst=f'{ROOT}/models/{sub}/{fn}'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if os.path.exists(dst) and os.path.getsize(dst) > 10_000_000:
        print('skip', fn); continue
    ok=False
    for url in urls:
        print('get', fn, 'from', url)
        if os.system(f'wget -q -c -O "{dst}" "{url}"') == 0 and os.path.getsize(dst) > 10_000_000:
            ok=True; break
    if not ok:
        print('FAILED all mirrors -> search this filename on huggingface.co and wget it to', dst)
print('model pass done')


In [ ]:
# 5) ReActor (face swap) for the stills workflow
import os
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-ReActor'):
    !git clone --depth 1 https://github.com/Gourieff/ComfyUI-ReActor.git /content/ComfyUI/custom_nodes/ComfyUI-ReActor
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-ReActor/requirements.txt
print('reactor ok')


In [ ]:
# 6) Workflows + Aarya refs -> ComfyUI input/
# Option 1: they are on your Drive (upload the aarya/ folder from the repo once):
import shutil, glob, os
src_wf=f'{ROOT}/workflows'
if glob.glob(f'{src_wf}/*.json'):
    os.makedirs('/content/aarya_workflows', exist_ok=True)
    for f in glob.glob(f'{src_wf}/*.json'):
        shutil.copy(f, '/content/aarya_workflows/')
    print('workflows at /content/aarya_workflows (you can also just drag the JSON into the ComfyUI tab)')
else:
    print('NO workflows on Drive: upload aarya/workflows/*.json to Drive/MyDrive/aarya/workflows')
    print('(download them from the Aarya Studio preview or git clone the repo on your laptop first)')
for f in glob.glob(f'{ROOT}/input/*'):
    shutil.copy(f, '/content/ComfyUI/input/')
print('input files:', os.listdir('/content/ComfyUI/input'))


In [ ]:
# 7) Launch ComfyUI + public tunnel URL
import subprocess, time, re, os
os.chdir('/content/ComfyUI')
log=open('/content/comfy.log','w')
subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188','--disable-auto-launch'],
                 stdout=log, stderr=subprocess.STDOUT)
if not os.path.exists('/content/cloudflared'):
    subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/content/cloudflared'])
    os.chmod('/content/cloudflared', 0o755)
tlog=open('/content/tunnel.log','w')
subprocess.Popen(['/content/cloudflared','tunnel','--no-autoupdate','--url','http://localhost:8188'],
                 stdout=tlog, stderr=subprocess.STDOUT)
url=None
for _ in range(60):
    time.sleep(2)
    txt=open('/content/tunnel.log').read()
    m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m: url=m.group(0); break
print('COMFYUI PUBLIC URL:', url or 'not up yet - re-run this cell / check /content/comfy.log')


In [ ]:
# 8) Save outputs to Drive (run after generating; also before ending session)
import shutil, os, glob
os.makedirs(f'{ROOT}/output', exist_ok=True)
for f in glob.glob('/content/ComfyUI/output/**/*', recursive=True):
    if os.path.isfile(f):
        dst=f.replace('/content/ComfyUI/output', f'{ROOT}/output')
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(f, dst)
print('outputs on Drive:', glob.glob(f'{ROOT}/output/**/*', recursive=True)[-5:])


## What to load per tier
- **TIER A:** `aarya_video_wan_animate2_gpu.json` (or `_distilled_`) — refs: `aarya_ref_front_bust.png`
  (bust drives) / `aarya_ref_fullbody.png` (full-body drives). `WanAnimate2Cache` already on `cpu`.
- **TIER B:** `aarya_stills_facelock_cpu.json` (runs on GPU here, just faster) — source face: `aarya_face_direct.png`.
- Identity markers to QA: black bindi between brows · silver stud LEFT nostril · centre-parted jet-black waves.

## If something breaks
- 404 on a model URL → the HF path moved; search the filename on huggingface.co and re-run cell 4.
- Tunnel died → re-run cell 7.
- Session recycled → re-run cells 2-7 (models come from Drive, no re-download).
